# Warsaw apartment asking prices — exploratory analysis

Run the preprocessing step first. Price/feature exploration below uses the training partition only. Fictional demo data verifies the code; it does not describe the actual Warsaw market.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Open this notebook from the project folder.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.preprocess import clean_listings
from src.train import split_data

plt.rcParams.update({'figure.figsize': (9, 4), 'axes.grid': True})

In [ ]:
path = ROOT / 'data' / 'training.csv'
if not path.exists():
    raise FileNotFoundError('Run python -m tests.demo_data and python -m src.preprocess --input data/demo_observations.csv first.')
raw = pd.read_csv(path, dtype={'source': 'string', 'listing_id': 'string'})
clean = clean_listings(raw, keep='earliest')
print('Input rows:', len(raw), 'Valid unique listings:', len(clean))
print('Sources:', sorted(clean['source'].unique()))
if clean['source'].str.contains('demo', case=False).any():
    print('FICTIONAL DEMONSTRATION DATA — not real market evidence.')
clean[['source', 'listing_id', 'observed_at']].head()

## Preserve the holdouts

The default split separates whole observation days. If only an initial snapshot is available, explicitly change `SPLIT_MODE` to `group` for exploration. Do not use test price distributions to choose features or repeatedly tune the model.

In [ ]:
SPLIT_MODE = 'temporal'
train, validation, test, split_metadata = split_data(clean, mode=SPLIT_MODE)
eda = train.copy()
print('Train / validation / test:', len(train), len(validation), len(test))
print('Split type:', split_metadata['type'])
print('Training date range:', eda['observed_at'].min(), 'to', eda['observed_at'].max())
print('All following price/feature plots use TRAIN only.')
eda[['price_pln', 'area_m2', 'rooms', 'distance_km', 'price_per_m2']].describe()

In [ ]:
missing_pct = eda.isna().mean().mul(100).sort_values(ascending=False)
print(missing_pct[missing_pct > 0].round(1).to_string())
missing_pct[missing_pct > 0].plot.bar(title='Missing values in training data (%)', ylabel='%')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
eda['price_pln'].div(1_000_000).plot.hist(bins=20, ax=axes[0], title='Asking price — training')
axes[0].set_xlabel('Million PLN')
eda['price_per_m2'].plot.hist(bins=20, ax=axes[1], title='Price per m² — diagnostic only')
axes[1].set_xlabel('PLN / m²')
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(eda['area_m2'], eda['price_pln'].div(1_000_000), alpha=0.65)
ax.set(xlabel='Area (m²)', ylabel='Asking price (million PLN)', title='Area and asking price — training')
fig.tight_layout()
plt.show()
print(eda.groupby('district', dropna=False).agg(listings=('listing_id', 'count'), median_price_per_m2=('price_per_m2', 'median')).round(0).sort_values('listings', ascending=False).to_string())

## Read a completed evaluation

MAE and RMSE below are in PLN. Compare the model against the median baseline on the same test partition. A minimum split is a software guard, not proof that a dataset is representative. Cross-source/relisted apartments and missing quality features still need investigation.

In [ ]:
metrics_path = ROOT / 'models' / 'metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('Reported split:', metrics['split']['type'])
    print('XGBoost test:', metrics['xgboost'])
    print('Median baseline test:', metrics['baseline_median'])
else:
    print('Run python -m src.train to produce an evaluation report.')

## Findings to write after collecting real data

- How many new listings and observation days are available?
- Which features are missing or suspicious?
- How does error vary by district, area and observation period?
- Does XGBoost beat the baseline on later observations?
- Could property condition, duplicate listings or inactive offers explain apparent discounts?

Keep actual conclusions separate from fictional demo results. An advertised asking price is not a completed transaction price.